In [ ]:
%%sql -r dataframe_11
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS spcs_test_role;

CREATE DATABASE IF NOT EXISTS spcs_tutorial_db;

GRANT CREATE POSTGRES INSTANCE ON ACCOUNT TO ROLE spcs_test_role;

CREATE OR REPLACE WAREHOUSE spcs_tutorial_warehouse WITH
  WAREHOUSE_SIZE='X-SMALL';
GRANT USAGE ON WAREHOUSE spcs_tutorial_warehouse TO ROLE spcs_test_role;

GRANT BIND SERVICE ENDPOINT ON ACCOUNT TO ROLE spcs_test_role;

CREATE COMPUTE POOL IF NOT EXISTS spcs_tutorial_compute_pool
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = CPU_X64_XS;
GRANT USAGE, MONITOR ON COMPUTE POOL spcs_tutorial_compute_pool TO ROLE spcs_test_role;

GRANT ROLE spcs_test_role TO USER "ユーザー名";

CREATE SCHEMA IF NOT EXISTS data_schema;


-- select SYSTEM$GET_SNOWFLAKE_EGRESS_IP_RANGES()でsnowflakeのiprangeがみれる
CREATE OR REPLACE NETWORK RULE spcs_tutorial_db.data_schema.spcs_test_pg_ingress_from_snowflake
  MODE = POSTGRES_INGRESS
  TYPE = IPV4
  VALUE_LIST = (
    '0.0.0.0/0'
  );

CREATE OR REPLACE NETWORK POLICY SPCS_TEST_PG_NETWORK_POLICY
  ALLOWED_NETWORK_RULE_LIST = (
    'spcs_tutorial_db.data_schema.spcs_test_pg_ingress_from_snowflake'
  );

GRANT USAGE ON NETWORK POLICY SPCS_TEST_PG_NETWORK_POLICY TO ROLE spcs_test_role;

-- 表示されるhostとsnowflake_adminの情報はメモする
CREATE POSTGRES INSTANCE spcs_test_pg_instance
  COMPUTE_FAMILY = 'BURST_S'
  STORAGE_SIZE_GB = 10
  AUTHENTICATION_AUTHORITY = POSTGRES
  NETWORK_POLICY = 'SPCS_TEST_PG_NETWORK_POLICY'
;

In [ ]:
%%sql -r dataframe_15
USE ROLE ACCOUNTADMIN;
USE DATABASE spcs_tutorial_db;
USE SCHEMA data_schema;

-- spcsが使う Postgres egress 用ネットワークルール
CREATE OR REPLACE NETWORK RULE spcs_tutorial_db.data_schema.spcs_test_pg_egress
  TYPE = HOST_PORT
  MODE = EGRESS
  VALUE_LIST = ('取得したhost名:5432');

-- spcsが使う Postgres 用 EAI
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION spcs_test_pg_eai
  ALLOWED_NETWORK_RULES = (spcs_tutorial_db.data_schema.spcs_test_pg_egress)
  ENABLED = TRUE;

GRANT USAGE ON INTEGRATION spcs_test_pg_eai TO ROLE spcs_test_role;


GRANT OWNERSHIP ON DATABASE spcs_tutorial_db TO ROLE spcs_test_role COPY CURRENT GRANTS;
GRANT OWNERSHIP ON SCHEMA spcs_tutorial_db.data_schema TO ROLE spcs_test_role COPY CURRENT GRANTS;

In [ ]:
%%sql -r dataframe_2
USE ROLE spcs_test_role;
USE DATABASE spcs_tutorial_db;
USE WAREHOUSE spcs_tutorial_warehouse;

CREATE SCHEMA IF NOT EXISTS data_schema;
USE SCHEMA data_schema;

CREATE IMAGE REPOSITORY IF NOT EXISTS spcs_tutorial_repository;
CREATE STAGE IF NOT EXISTS spcs_tutorial_stage
  DIRECTORY = ( ENABLE = true );


SHOW COMPUTE POOLS;
SHOW WAREHOUSES;
SHOW IMAGE REPOSITORIES;
SHOW STAGES;

In [ ]:
%%sql -r dataframe_6
USE ROLE spcs_test_role;
USE DATABASE spcs_tutorial_db;
USE SCHEMA data_schema;
USE WAREHOUSE spcs_tutorial_warehouse;

DESCRIBE COMPUTE POOL spcs_tutorial_compute_pool;

CREATE SERVICE echo_service
  IN COMPUTE POOL spcs_tutorial_compute_pool
  FROM SPECIFICATION $$
    spec:
      containers:
      - name: echo
        image: /spcs_tutorial_db/data_schema/spcs_tutorial_repository/my_echo_service_image:latest
        env:
          SERVER_PORT: 8000
          CHARACTER_NAME: Bob
          POSTGRES_URL: postgresql://snowflake_admin:取得したpassword@取得したhost名:5432/postgres?sslmode=require
        readinessProbe:
          port: 8000
          path: /healthcheck
      endpoints:
      - name: echoendpoint
        port: 8000
        public: true
      $$
   EXTERNAL_ACCESS_INTEGRATIONS = (spcs_test_pg_eai)
   MIN_INSTANCES=1
   MAX_INSTANCES=1;

SHOW SERVICES;
DESC SERVICE echo_service;


In [ ]:
%%sql -r dataframe_4
SHOW SERVICE CONTAINERS IN SERVICE echo_service;

In [ ]:
%%sql -r dataframe_3
SHOW ENDPOINTS IN SERVICE echo_service;

In [ ]:
%%sql -r dataframe_7
-- =========================
-- Cleanup for Snowflake SPCS tutorial resources
-- =========================

USE ROLE ACCOUNTADMIN;

USE DATABASE spcs_tutorial_db;
USE SCHEMA data_schema;

-- postgres 削除
DROP POSTGRES INSTANCE IF EXISTS spcs_test_pg_instance;

-- postgresのnetwork policy削除
DROP NETWORK POLICY IF EXISTS SPCS_TEST_PG_NETWORK_POLICY;
DROP NETWORK RULE IF EXISTS spcs_tutorial_db.data_schema.spcs_test_pg_ingress_from_snowflake;

--service削除
DROP SERVICE IF EXISTS echo_service;

-- spcsの外部統合削除
DROP EXTERNAL ACCESS INTEGRATION IF EXISTS spcs_test_pg_eai;
DROP NETWORK RULE IF EXISTS spcs_tutorial_db.data_schema.spcs_test_pg_egress;

-- Compute Pool停止 & 削除
ALTER COMPUTE POOL spcs_tutorial_compute_pool STOP ALL;

DROP COMPUTE POOL IF EXISTS spcs_tutorial_compute_pool;

-- Warehouse削除
DROP WAREHOUSE IF EXISTS spcs_tutorial_warehouse;

-- Image Repository削除
DROP IMAGE REPOSITORY IF EXISTS spcs_tutorial_repository;

-- Stage削除
DROP STAGE IF EXISTS spcs_tutorial_stage;

-- Schema削除
DROP SCHEMA IF EXISTS data_schema;

-- Database削除
DROP DATABASE IF EXISTS spcs_tutorial_db;

-- Roleをユーザーから剥奪
REVOKE ROLE spcs_test_role FROM USER MAHIRO_KAJIYAMA;

-- Role削除
DROP ROLE IF EXISTS spcs_test_role;